In [1]:
import numpy as np
import pandas as pd

from exerpy import ExergyAnalysis
from exerpy.analyses import _load_json

In [2]:
def diff_between_simulators(testcase):
    simulator_results = []
    for path in testcase.values():
        contents = _load_json(path)
        if "settings" not in contents:
            contents["settings"] = {}

        simulator_results += [
            ExergyAnalysis.from_json(path, **contents["settings"])
        ]

    sim1 = simulator_results[0]
    sim2 = simulator_results[1]

    columns = ["m", "p", "T"]
    if sim1.chemExLib is not None:
        columns.append("e_CH")
    if sim1.split_physical_exergy:
        columns.append("e_M")
        columns.append("e_T")
    else:
        columns.append("e_PH")

    df_sim1 = pd.DataFrame.from_dict(
        sim1._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")
    df_sim2 = pd.DataFrame.from_dict(
        sim2._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")

    overlapping_index = list(
        set(df_sim1.index.tolist()) & set(df_sim2.index.tolist())
    )
    df_sim1 = df_sim1.loc[overlapping_index].round(6)
    df_sim2 = df_sim2.loc[overlapping_index].round(6)

    # inf means that sim2 has 0 value, comparison does not make sense there
    # and sometimes there seem to be NaN values in the dataframes, those are
    # removed as well
    return (
        (df_sim1 - df_sim2) / df_sim2
    ).abs().replace(np.inf, 0).fillna(0)

In [3]:
testcase = {
    "ebsilon": "ccpp_ebs.json",
    "tespy": "ccpp_tespy.json"
}
diff_between_simulators(testcase)


Component type 'Condenser' is deprecated and is mapped to 'HeatExchanger'. Dissipative behavior is now inferred from the temperature case or the E_L specification.


,m,p,T,e_CH,e_PH
2,0.000159,0.000000e+00,1.073468e-03,0.000830,2.062225e-03
13,0.000601,0.000000e+00,0.000000e+00,0.000000,3.205608e-05
10,0.000605,0.000000e+00,9.288467e-05,0.000000,5.951807e-05
11,0.000051,0.000000e+00,9.288467e-05,0.000000,5.951807e-05
14,0.000665,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
17,0.000601,0.000000e+00,0.000000e+00,0.000000,1.303753e-08
4,0.000162,0.000000e+00,3.513333e-08,0.000169,7.960676e-04
1,0.000159,0.000000e+00,0.000000e+00,0.000830,0.000000e+00
15,0.000665,0.000000e+00,4.135862e-06,0.000000,1.904954e-04
18,0.000051,0.000000e+00,2.257194e-09,0.000000,1.057542e-09


In [4]:
testcase = {
    "tespy": "ccpp_tespy.json",
    "aspen": "ccpp_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_CH,e_PH
2,0.000690,0.0,4.701387e-04,0.000829,1.712372e-03
13,0.000307,0.0,0.000000e+00,0.000000,4.785162e-05
10,0.000227,0.0,1.367758e-04,0.000000,8.830424e-05
11,0.000076,0.0,1.367758e-04,0.000000,8.830424e-05
14,0.000258,0.0,0.000000e+00,0.000000,0.000000e+00
17,0.000307,0.0,2.123432e-07,0.000000,4.365327e-06
4,0.000710,0.0,7.026667e-10,0.001740,9.740188e-04
1,0.000690,0.0,0.000000e+00,0.000829,0.000000e+00
15,0.000258,0.0,4.135879e-06,0.000000,1.912064e-04
18,0.000076,0.0,0.000000e+00,0.000000,6.643333e-07


In [5]:
testcase = {
    "ebsilon": "ccpp_ebs.json",
    "aspen": "ccpp_aspen.json"
}
diff_between_simulators(testcase)

Component type 'Condenser' is deprecated and is mapped to 'HeatExchanger'. Dissipative behavior is now inferred from the temperature case or the E_L specification.


,m,p,T,e_CH,e_PH
2,0.000849,0.000000e+00,6.038342e-04,2.349319e-07,3.533845e-04
13,0.000908,0.000000e+00,0.000000e+00,0.000000e+00,1.579400e-05
10,0.000831,0.000000e+00,4.387841e-05,0.000000e+00,2.878092e-05
19,0.000365,0.000000e+00,8.053258e-05,0.000000e+00,5.713654e-04
11,0.000025,0.000000e+00,4.387841e-05,0.000000e+00,2.878092e-05
14,0.000924,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
17,0.000908,0.000000e+00,2.123432e-07,0.000000e+00,4.352289e-06
4,0.000871,0.000000e+00,3.443066e-08,1.909568e-03,1.771758e-04
1,0.000849,0.000000e+00,0.000000e+00,2.349319e-07,0.000000e+00
15,0.000924,0.000000e+00,0.000000e+00,0.000000e+00,6.745670e-07


In [6]:
# Load the CSV files
tespy_results = pd.read_csv("ccpp_components_tespy.csv", index_col=0)
ebsilon_results = pd.read_csv("ccpp_components_ebsilon.csv", index_col=0)

# Step 1: Normalize the component names in Tespy
# Group "drum" and "drum pump" into "EVA"
tespy_results.loc[tespy_results["Component"].isin(["drum", "drum pump"]), "Component"] = "EVA"

# Group "dea steam inlet valve" into "DEA"
tespy_results.loc[tespy_results["Component"] == "dea steam inlet valve", "Component"] = "DEA"

# Group "MIX" into "DEA"
ebsilon_results.loc[ebsilon_results["Component"] == "MIX", "Component"] = "DEA"

# Aggregate the values by summing them up
tespy_results = tespy_results.groupby("Component", as_index=False).sum(numeric_only=True)
ebsilon_results = ebsilon_results.groupby("Component", as_index=False).sum(numeric_only=True)

# Step 2: Merge DataFrames on Component name
merged_df = pd.merge(ebsilon_results, tespy_results, on="Component", suffixes=("_Ebsilon", "_Tespy"))

# Step 3: Compute Differences
columns_to_compare = ["E_F [kW]", "E_P [kW]", "E_D [kW]", "E_L [kW]", "epsilon [%]", "y [%]", "y* [%]"]
for col in columns_to_compare:
    merged_df[f"Diff_{col}"] = merged_df[f"{col}_Tespy"] - merged_df[f"{col}_Ebsilon"]

# Step 4: Select only the relevant columns
diff_columns = [col for col in merged_df.columns if col.startswith("Diff_E_D") or col.endswith("E_D [kW]_Ebsilon") or col.endswith("E_D [kW]_Tespy")]
diff_df = merged_df[["Component"] + diff_columns].copy()
diff_df["Relative error [%]"] = diff_df["Diff_E_D [kW]"] / diff_df["E_D [kW]_Tespy"] * 100

# Display the filtered DataFrame
diff_df

,Component,E_D [kW]_Ebsilon,E_D [kW]_Tespy,Diff_E_D [kW],Relative error [%]
0,CC,195766.602870,195534.636129,-231.966741,-0.118632
1,COMP,10945.496471,10953.652110,8.155639,0.074456
2,COND,3717.826764,2335.818019,-1382.008745,-59.165942
3,DEA,3184.745810,3183.009812,-1.735997,-0.054539
4,ECO,1475.374949,1474.821192,-0.553757,-0.037547
5,EVA,11648.296333,11650.581566,2.285233,0.019615
6,GEN1,3777.417074,3777.770170,0.353096,0.009347
7,GEN2,798.734685,798.438835,-0.295850,-0.037054
8,GT,15969.551601,15984.355276,14.803676,0.092614
9,HC,20.937228,0.000000,-20.937228,-inf
